# 03 MLP 分类

## 本节目标

本节从线性回归过渡到分类任务，重点理解 MLP 如何建模非线性边界。

你将完成：

- 构造二维非线性分类数据。
- 理解 logits、label、`CrossEntropyLoss` 的配合方式。
- 训练一个 MLP 分类器。
- 绘制 loss 曲线和决策区域。
- 总结分类任务常见调试点。

## 背景问题

本节关注的问题是：当两类样本无法用一条直线分开时，模型如何学习弯曲的分类边界？

MLP 的关键在于“线性层 + 非线性激活”。如果没有激活函数，多层线性层叠加后仍然等价于一个线性变换。

## 核心概念

- **MLP**：多层感知机，由多个全连接层和激活函数组成。
- **ReLU**：常用激活函数，引入非线性。
- **Logits**：模型输出的未归一化类别分数。
- **CrossEntropyLoss**：分类任务常用损失，输入 logits 和类别索引。
- **Decision boundary**：模型把不同类别分开的边界。

初学者容易把 logits 和概率混在一起。训练时通常直接把 logits 传给 `CrossEntropyLoss`，不需要手动 softmax。

## 最小代码示例：准备非线性数据

这里使用 `make_moons` 生成二维数据。它的类别边界是弯曲的，适合观察 MLP 的非线性建模能力。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

x, y = make_moons(n_samples=600, noise=0.18, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype(np.float32)
x_test = scaler.transform(x_test).astype(np.float32)

x_train = torch.tensor(x_train)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test)
y_test = torch.tensor(y_test, dtype=torch.long)

plt.figure(figsize=(5, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="coolwarm", s=18)
plt.title("Training data")
plt.grid(alpha=0.3)
plt.show()

print("x_train:", x_train.shape)
print("y_train:", y_train.shape, y_train.dtype)

## 最小代码示例：logits 与 CrossEntropyLoss

`CrossEntropyLoss` 期望输入形状为 `[batch, num_classes]` 的 logits，标签形状通常是 `[batch]`，类型是 `torch.long`。

In [ ]:
toy_logits = torch.tensor([[2.0, 0.5], [0.1, 1.2], [1.0, 0.8]])
toy_labels = torch.tensor([0, 1, 0], dtype=torch.long)

loss_fn = nn.CrossEntropyLoss()
toy_loss = loss_fn(toy_logits, toy_labels)

print("logits shape:", toy_logits.shape)
print("labels shape:", toy_labels.shape)
print("toy loss:", toy_loss.item())
print("predicted classes:", toy_logits.argmax(dim=1))

## 完整实验：定义 MLP

这个模型输入二维点，输出两个类别的 logits。隐藏层宽度不需要很大，因为这个任务本身较小。

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


def accuracy(logits, labels):
    predictions = logits.argmax(dim=1)
    return (predictions == labels).float().mean().item()


model = MLP(hidden_dim=32)
print(model)

## 完整实验：训练与评估

训练循环和线性回归类似，只是 loss 换成了分类损失，评价指标增加了 accuracy。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

train_losses = []
train_accs = []
test_accs = []

for epoch in range(150):
    model.train()
    logits = model(x_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        train_acc = accuracy(model(x_train), y_train)
        test_acc = accuracy(model(x_test), y_test)

    train_losses.append(loss.item())
    train_accs.append(train_acc)
    test_accs.append(test_acc)

print(f"final loss={train_losses[-1]:.4f}")
print(f"train acc={train_accs[-1]:.3f}")
print(f"test acc={test_accs[-1]:.3f}")

## 实验观察：loss 与 accuracy

从运行结果可以看到，loss 下降时 accuracy 通常会上升。但二者不是完全等价的：loss 关注预测分数的置信程度，accuracy 只关注类别是否预测正确。

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross entropy")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_accs, label="train acc")
plt.plot(test_accs, label="test acc")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 实验观察：决策区域

决策区域可以帮助我们直观看到模型学到的分类边界。对于 moon 数据，合理的边界通常是弯曲的。

In [ ]:
xx, yy = np.meshgrid(
    np.linspace(x_train[:, 0].min() - 0.6, x_train[:, 0].max() + 0.6, 180),
    np.linspace(x_train[:, 1].min() - 0.6, x_train[:, 1].max() + 0.6, 180),
)
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()].astype(np.float32))

model.eval()
with torch.no_grad():
    zz = model(grid).argmax(dim=1).numpy().reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
plt.scatter(x_test[:, 0], x_test[:, 1], c=y_test, cmap="coolwarm", s=18, edgecolor="k")
plt.title("Decision regions on test points")
plt.grid(alpha=0.3)
plt.show()

## 常见错误

1. **标签 dtype 错误**：`CrossEntropyLoss` 的 label 应为 `torch.long`。  
2. **提前 softmax**：不要把 softmax 后的概率传给 `CrossEntropyLoss`。  
3. **输出类别数不匹配**：最后一层输出维度应等于类别数。  
4. **没有激活函数**：多层线性层仍然表达有限。  
5. **训练/评估模式混乱**：评估时使用 `model.eval()` 和 `torch.no_grad()`。

调试时可以优先检查：`logits.shape`、`labels.dtype`、loss 是否下降、训练集和测试集 accuracy 是否差距过大。

## 概念问答

**Q：MLP 为什么能学习非线性边界？**  
A：因为激活函数打破了线性层叠加仍是线性的限制。

**Q：为什么最后一层不用 ReLU？**  
A：分类 logits 可以是任意实数，后续交给 `CrossEntropyLoss` 处理。

**Q：测试集 accuracy 比训练集低很多说明什么？**  
A：可能存在过拟合，也可能数据划分较难或训练样本不足。

**Q：隐藏层越大越好吗？**  
A：不一定。更大的模型可能更强，也可能更容易过拟合。

## 小结

MLP 是理解神经网络非线性表达的关键入口。掌握 logits、label、loss 和决策边界后，再学习 CNN、RNN 和 Transformer 会更顺畅。